# Human-Like Anaphor Resolution in Large Language Models

This notebook rebuilds the paper surprisal tables and figures for `cogsci_2026.pdf`.

Set `DATA_SOURCE` in the config cell to one of:

- `'published'`: load the checked-in `data/exp1.csv`, `data/exp2.csv`, `data/exp3.csv` (paper reference data; fast).
- `'generated'`: load the previously-saved `generated/data/exp*.csv` files (lets you re-plot without rerunning the models).
- `'regenerate'`: recompute surprisals from raw passage files using Hugging Face models through `paper_pipeline.py` (slow; requires model access and enough memory).

Use `EXPERIMENTS_TO_RUN` to run one experiment or all three independently. Figures are written to `generated/figures/`.


In [ ]:
%%capture
%pip install -r requirements.txt
# sanity check
%python -c "import transformers, huggingface_hub as h; print('transformers', transformers.__version__, 'hub', h.__version__)"

In [ ]:
import os
os.environ["HF_HOME"] = "./hf"
os.environ["HF_HUB_CACHE"] = "./hf/hub"
os.environ.setdefault("HF_HOME", "./hf")
os.environ.setdefault("HF_HUB_CACHE", "./hf/hub")
os.environ.setdefault("HF_DATASETS_CACHE", "./hf/datasets")
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
import re
import json
from huggingface_hub import login

In [ ]:
# from huggingface_hub import notebook_login
# # huggingface_hub 0.36.2 uses notebook_login(..., new_session=False) in notebooks.
# notebook_login(new_session=False)


In [ ]:
# from huggingface_hub import notebook_login
# notebook_login(new_session=False)


In [ ]:
# Optional: check Hugging Face authentication before using gated/private models.
import shutil
import subprocess

if shutil.which('hf'):
    subprocess.run(['hf', 'auth', 'whoami'], check=False)
else:
    print('Hugging Face CLI is not installed; install requirements or run notebook_login() if needed.')


In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display

import pandas as pd

CANDIDATE_ROOTS = [Path.cwd(), Path.cwd() / 'anaphor', Path.cwd().parent / 'anaphor']
ROOT = next((path.resolve() for path in CANDIDATE_ROOTS if (path / 'paper_pipeline.py').exists()), Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from paper_pipeline import (
    EXPERIMENTS,
    MODEL_ORDER,
    REPO_TO_PAPER_EXPERIMENT,
    register_hf_model,
    run_experiment,
    plot_surprisal_figure,
)
from comprehension_pipeline import (
    GeminiComprehensionScorer,
    calculate_passage_accuracy,
    load_correct_answers,
    load_model_responses,
    load_scored_comprehension,
    PAPER_COMPREHENSION_MODEL_ORDER,
    REPO_TO_PAPER_COMPREHENSION_STUDY,
    REPO_STUDY_TO_COMPREHENSION_FIGURE,
    save_comprehension_outputs,
    score_model_responses,
    summarize_accuracy_by_version,
)

GENERATED_DATA_DIR = ROOT / 'generated' / 'data'
GENERATED_FIGURE_DIR = ROOT / 'generated' / 'figures'
GENERATED_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
ROOT


In [ ]:
# Choose how to load the surprisal tables.
#   'published'  -> data/exp*.csv (paper reference)
#   'generated'  -> generated/data/exp*.csv (saved previous run)
#   'regenerate' -> recompute from raw passages with Hugging Face models (slow)
DATA_SOURCE = 'published'

# Run any subset of repo experiments. Paper numbering maps as:
#   paper Experiment 1 -> repo exp3
#   paper Experiment 2 -> repo exp1
#   paper Experiment 3 -> repo exp2
EXPERIMENTS_TO_RUN = ['exp3', 'exp1', 'exp2']

# Aliases from paper_pipeline.py and direct Hugging Face repo ids are both accepted.
# Examples for newer models:
#   MODELS_TO_RUN = ['GPT2', 'meta-llama/Llama-3.2-1B', 'Qwen/Qwen2.5-1.5B']
#   register_hf_model('Qwen2.5-1.5B', 'Qwen/Qwen2.5-1.5B', torch_dtype='auto')
MODELS_TO_RUN = MODEL_ORDER

# Only used when DATA_SOURCE == 'regenerate'.
# Set to True to overwrite generated/data/*.csv with freshly recomputed outputs.
WRITE_GENERATED_DATA = False

# Leave as None to plot paper models first, followed by any extra regenerated models.
PLOT_MODEL_ORDER = None

DATA_SOURCE, EXPERIMENTS_TO_RUN


In [ ]:
experiment_results = {}
wide_tables = {}
validation_rows = []

def run_selected_experiment(experiment_name):
    if experiment_name not in EXPERIMENTS_TO_RUN:
        print(f'Skipping {experiment_name}')
        return None

    result = run_experiment(
        ROOT,
        experiment_name,
        data_source=DATA_SOURCE,
        model_names=MODELS_TO_RUN,
        generated_data_dir=GENERATED_DATA_DIR,
        figure_dir=GENERATED_FIGURE_DIR,
        write_generated_data=WRITE_GENERATED_DATA,
        write_figures=False,
        plot_model_order=PLOT_MODEL_ORDER,
    )
    experiment_results[experiment_name] = result
    wide_tables[experiment_name] = result.wide_df
    validation_rows.append({
        'repo_experiment': experiment_name,
        'paper_experiment': f"Experiment {REPO_TO_PAPER_EXPERIMENT[experiment_name]}",
        'source': result.source,
        'rows': int(result.wide_df.shape[0]),
        'columns': int(result.wide_df.shape[1]),
        'saved_csv': str(result.csv_path.relative_to(ROOT)) if result.csv_path and result.csv_path.exists() else '(not written)',
        'saved_long_csv': str(result.long_csv_path.relative_to(ROOT)) if result.long_csv_path and result.long_csv_path.exists() else '(not written)',
        'max_abs_diff_vs_reference': None if result.validation is None else result.validation.get('max_abs_diff'),
    })
    return result

# Each repo experiment is intentionally independent; edit EXPERIMENTS_TO_RUN above.
exp3_result = run_selected_experiment('exp3')  # paper Experiment 1
exp1_result = run_selected_experiment('exp1')  # paper Experiment 2
exp2_result = run_selected_experiment('exp2')  # paper Experiment 3

validation_df = pd.DataFrame(validation_rows)
display(validation_df)


In [ ]:
figure_rows = []

for experiment_name, result in experiment_results.items():
    png_path = plot_surprisal_figure(
        experiment_name,
        result.wide_df,
        GENERATED_FIGURE_DIR,
        model_order=PLOT_MODEL_ORDER,
    )
    result.figure_png_path = png_path
    figure_rows.append({
        'experiment': experiment_name,
        'png': str(png_path.relative_to(ROOT)),
    })

figures_df = pd.DataFrame(figure_rows)
display(figures_df)


In [ ]:
for experiment_name in EXPERIMENTS_TO_RUN:
    if experiment_name not in wide_tables:
        continue
    print(f'===== {experiment_name} =====')
    display(wide_tables[experiment_name].head())


## Comprehension Scoring and Figures

This section incorporates the automated comprehension-scoring workflow from `anaphor-automate-comprehesion-scoring`. By default it loads the already judged CSVs and regenerates accuracy summaries/figures. Summary CSVs are written to `data/`, and comprehension figures are written as PNG files in `generated/figures/`. Set `RUN_COMPREHENSION_RESCORING = True` only when you want to call the judge model again.


In [ ]:
COMPREHENSION_STUDIES = [3, 1, 2]  # paper Experiments 1, 2, 3 respectively
COMPREHENSION_SCORES_DIR = ROOT / 'anaphor-automate-comprehesion-scoring' / 'results' / 'automate_scores'
COMPREHENSION_MODEL_RESPONSES_DIR = ROOT / 'anaphor-automate-comprehesion-scoring' / 'results' / 'model_responses'
COMPREHENSION_DATA_DIR = ROOT / 'data'
COMPREHENSION_FIGURE_DIR = GENERATED_FIGURE_DIR
COMPREHENSION_DATA_DIR.mkdir(parents=True, exist_ok=True)
COMPREHENSION_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Leave this False for reproducible, no-API Run All behavior.
RUN_COMPREHENSION_RESCORING = False
COMPREHENSION_JUDGE_MODEL = os.environ.get('COMPREHENSION_JUDGE_MODEL', 'gemini-2.5-flash')
COMPREHENSION_STUDIES


In [ ]:
comprehension_scores = {}
comprehension_inventory = []

for study_num in COMPREHENSION_STUDIES:
    scored_df = load_scored_comprehension(ROOT, study_num)
    comprehension_scores[study_num] = scored_df
    paper_models_present = [model for model in PAPER_COMPREHENSION_MODEL_ORDER if model in set(scored_df['model'])] if not scored_df.empty else []
    comprehension_inventory.append({
        'paper_experiment': f"Experiment {REPO_TO_PAPER_COMPREHENSION_STUDY[study_num]}",
        'repo_study': study_num,
        'rows': int(scored_df.shape[0]),
        'scored_models': ', '.join(sorted(scored_df['model'].dropna().unique())) if not scored_df.empty else '',
        'plotted_models': ', '.join(paper_models_present),
        'versions': ', '.join(sorted(scored_df['version'].dropna().unique())) if not scored_df.empty else '',
    })

comprehension_inventory_df = pd.DataFrame(comprehension_inventory)
display(comprehension_inventory_df)


In [ ]:
comprehension_figure_rows = []
comprehension_accuracy_tables = {}

for study_num, scored_df in comprehension_scores.items():
    if scored_df.empty:
        continue

    paper_experiment = REPO_TO_PAPER_COMPREHENSION_STUDY[study_num]
    figure_number = REPO_STUDY_TO_COMPREHENSION_FIGURE[study_num]
    prefix = f'fig{figure_number}_comprehension'
    paths = save_comprehension_outputs(
        scored_df,
        COMPREHENSION_DATA_DIR,
        prefix=prefix,
        figure_dir=COMPREHENSION_FIGURE_DIR,
    )
    paper_scored_df = scored_df[scored_df['model'].isin(PAPER_COMPREHENSION_MODEL_ORDER)].copy()
    comprehension_accuracy_tables[study_num] = summarize_accuracy_by_version(paper_scored_df)
    comprehension_figure_rows.append({
        'paper_experiment': f'Experiment {paper_experiment}',
        'figure': f'Figure {figure_number}',
        'repo_study': study_num,
        'csv': str(paths.by_model_version_csv.relative_to(ROOT)),
        'png': str(paths.model_version_png.relative_to(ROOT)),
    })

comprehension_figures_df = pd.DataFrame(comprehension_figure_rows)
display(comprehension_figures_df)


In [ ]:
for study_num, scored_df in comprehension_scores.items():
    if scored_df.empty:
        continue
    paper_scored_df = scored_df[scored_df['model'].isin(PAPER_COMPREHENSION_MODEL_ORDER)].copy()
    print(f'===== study {study_num} comprehension accuracy by model/version =====')
    display(comprehension_accuracy_tables[study_num])
    print(f'===== study {study_num} passage-level accuracy preview =====')
    display(calculate_passage_accuracy(paper_scored_df).head())


In [ ]:
if RUN_COMPREHENSION_RESCORING:
    judge = GeminiComprehensionScorer(model_name=COMPREHENSION_JUDGE_MODEL)
    rescored_outputs = []

    for study_num in COMPREHENSION_STUDIES:
        correct_answers = load_correct_answers(ROOT, study_num)
        response_dir = COMPREHENSION_MODEL_RESPONSES_DIR / f'study {study_num}'
        for response_path in sorted(response_dir.glob('*.csv')):
            responses_df = load_model_responses(ROOT, study_num, response_path)
            scored_df = score_model_responses(
                responses_df,
                correct_answers,
                study_num=study_num,
                scorer=judge,
            )
            output_dir = COMPREHENSION_DATA_DIR / 'rescored_comprehension' / f'study {study_num}'
            output_dir.mkdir(parents=True, exist_ok=True)
            output_path = output_dir / response_path.name
            scored_df.to_csv(output_path, index=False)
            rescored_outputs.append({
                'study': study_num,
                'model_file': response_path.name,
                'rows': int(scored_df.shape[0]),
                'output': str(output_path.relative_to(ROOT)),
            })

    display(pd.DataFrame(rescored_outputs))
else:
    print('Skipping API-backed comprehension rescoring. Set RUN_COMPREHENSION_RESCORING = True to enable it.')

## Notes

- `generated/data/exp*_long.csv` contains explicit `passage_id`, `version`, `model`, and `surprisal` columns, which is easier to publish than the legacy wide tables alone.
- The regeneration path in `paper_pipeline.py` preserves the quirks of the original notebooks where possible, including the experiment-specific target-word logic and the legacy row order used by the checked-in CSVs.